# Lab25. Seaborn 고급 시각화로 토마토 자료 분석

PPT를 보며 각 코드 셀의 `____`를 채우세요. 밑줄 전체를 지우고 값 또는 이름을 입력한 뒤 실행합니다. 문자열은 따옴표까지 입력하세요. 위에서부터 순서대로 실행합니다.

## 0. 한글 표시 준비

`____`에 `koreanize-matplotlib`을 입력하고 실행하세요.

In [ ]:
!pip install ____

## 1. 토마토 자료 준비하기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import koreanize_matplotlib

BASE = 'https://raw.githubusercontent.com/hongsukyi/Lectures/main/data/' 
tomato = pd.read_csv(BASE + ____, encoding='utf-8-sig')
cult = ['일반', '스테비아', '프리미엄']
tomato.head(3)

## 2. 재배방식별 당도 분포와 관측점

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(data=tomato, x='cultivation', 
               y='brix', order=cult,
               hue='cultivation', legend=False,
               inner=____, cut=0, ax=ax)
sns.swarmplot(data=tomato, x='cultivation', 
              y='brix', order=cult,
              color='black', alpha=.4, size=3, ax=ax)

ax.set_title('재배 방식별 당도 분포')
plt.show()

## 3. 재배방식별 당도 ANOVA

In [ ]:
groups = [tomato.loc[tomato['cultivation'] == c, ____] for c in cult]
anova = stats.f_oneway(*groups)

print(tomato.groupby('cultivation')['brix'].agg(['mean', 'std']).reindex(cult).round(2))
print(f'일원 ANOVA: F={anova.statistic:.2f}, p={anova.pvalue:.2e}')

## 4. Tukey 사후검정

In [ ]:
if anova.pvalue < ____:
    tukey = stats.tukey_hsd(*groups)
    pairs = []
    for i in range(len(cult)):
        for j in range(i + 1, len(cult)):
            pairs.append({
                '집단 1': cult[i], '집단 2': cult[j],
                '평균 차이': tukey.statistic[i, j],
                'p값': f'{tukey.pvalue[i, j]:.2e}'
            })
    print(pd.DataFrame(pairs).round({'평균 차이': 3}).to_string(index=False))
else:
    print('전체 차이가 확인되지 않아 사후검정을 생략합니다.')

## 5. ECDF로 누적분포 비교하기

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.ecdfplot(data=tomato, x=____, 
             hue='cultivation', 
             hue_order=cult,
             linewidth=2.3, ax=ax)

ax.axhline(0.5, color='gray', 
           linestyle='--', linewidth=1)
ax.set_title('재배 방식별 당도 누적분포 (ECDF)') 
plt.show()

## 6. Boxenplot으로 분포 비교하기

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.boxenplot(data=tomato, x='cultivation', 
              y=____, order=cult,
              hue='cultivation', legend=False, 
              ax=ax)
ax.set_title('재배 방식별 당도 분포 (Boxenplot)')
plt.show()

print(tomato.groupby('cultivation')['brix'].agg(
    ['min', 'max']).reindex(cult).round(2))

## 7. 저장 전후 자료를 긴 형태로 바꾸기

In [ ]:
paired = tomato[['id', 'brix', 'brix_after']].melt(
    id_vars=____, var_name='시점', value_name='당도')

paired['시점'] = paired['시점'].map(
    {'brix': '저장 전', 'brix_after': '저장 후'})

## 8. 같은 토마토의 저장 전후 변화

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for _, part in paired.groupby('id'):
    ax.plot(part['시점'], part['당도'], color='gray', alpha=.12, linewidth=.8)
sns.pointplot(data=paired, x='시점', y='당도', order=['저장 전', '저장 후'],
              errorbar=('ci', ____), capsize=.12, ax=ax)
ax.set_title('같은 토마토의 저장 전후 당도 변화')
plt.show()

## 9. 저장 전후 대응표본 t검정

In [ ]:
paired_t = stats.ttest_rel(tomato[____], tomato['brix'])
change = tomato['brix_after'] - tomato['brix']

print(f'평균 변화: {change.mean():.3f} Brix (표준편차 {change.std():.3f})')
print(f'대응 t검정: t = {paired_t.statistic:.3f}, p = {paired_t.pvalue:.2e}')

## 10. 일조량과 당도의 관계

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

r_sun = tomato['sunlight'].corr(tomato['brix'])
sns.regplot(data=tomato, x=____, y='brix', scatter_kws={'alpha': .4}, ax=ax)
ax.set_title(f'일조량과 당도의 관계 (r={r_sun:.3f})')
plt.show()

## 11. 재배방식별 회귀선

In [ ]:
g = sns.lmplot(data=tomato, x='weight', y='brix', 
               hue=____,
                hue_order=cult,
                height=4, aspect=1.2, 
               scatter_kws={'alpha': .5, 's': 40})

g.set_axis_labels('무게 (g)', '당도 (Brix)')
g.figure.suptitle('재배 방식별 무게-당도 회귀 관계', y=1.02)
plt.show()

## 12. 네 변수의 상관관계 히트맵

In [ ]:
corr = tomato[['brix', 'brix_after', 'sunlight', 'weight']].corr().round(2)
corr = corr.rename(
    index={'brix': '당도', 'brix_after': '저장후당도', 'sunlight': '일조량', 'weight': '무게'},
    columns={'brix': '당도', 'brix_after': '저장후당도', 'sunlight': '일조량', 'weight': '무게'})

fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(corr, annot=____, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title('수치 변수 상관관계')
plt.show()

## 13. Pairplot으로 전체 관계 보기

In [ ]:
numeric_cols = ['brix', 'brix_after', 
                'sunlight', 'weight']
g = sns.pairplot(tomato[numeric_cols
    + ['cultivation']],
                 hue='cultivation', 
                 hue_order=cult,
                 corner=____, 
                 diag_kind='kde', 
                 height=2.1,
                 plot_kws={'alpha':.6,'s':28})

g.figure.suptitle('토마토 수치 변수의 전체 관계', y=1.02)
plt.show()

## 14. Jointplot으로 결합밀도 보기

In [ ]:
g = sns.jointplot(
    data=tomato, 
    x='sunlight', 
    y='brix', kind=____,
    fill=True, cmap='mako', 
    thresh=.05, levels=10, 
    height=4.5)

sns.scatterplot(
    data=tomato, x='sunlight', 
    y='brix', hue='cultivation',
    hue_order=cult,
    s=30, alpha=.6, 
    edgecolor='white', 
    linewidth=.4, ax=g.ax_joint)

g.ax_joint.set_xlabel('일조량 (시간)')
g.ax_joint.set_ylabel('당도 (Brix)')
g.figure.suptitle('일조량과 당도의 결합 밀도', y=1.02)
plt.show()

## 심화 탐구 — 참고 정답

| 질문 | Seaborn을 사용한 이유 |
|---|---|
| 저장 후 당도가 변하는가? | 같은 토마토의 전후 변화를 선으로 연결하고, pointplot으로 평균과 Bootstrap 신뢰구간을 쉽게 표시할 수 있습니다. 변화의 검정은 대응표본 t검정으로 수행합니다. |
| 무게·일조량과 당도가 관계있는가? | regplot은 회귀선과 신뢰구간을 함께 그리고, lmplot은 hue로 집단별 회귀선을 자동으로 나눕니다. |
| 네 변수가 전체적으로 어떻게 얽혀 있는가? | heatmap은 상관계수를 색과 숫자로, pairplot은 변수 쌍의 관계를 격자로, jointplot은 두 변수의 관계와 주변 분포를 함께 보여줍니다. |